# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asadnaeem23/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!git clone https://github.com/Asadnaeem23/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 139 (delta 49), reused 89 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.88 MiB | 12.99 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [8]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I will use a Decision Tree for the Content Refresh lane.

A Decision Tree fits this lane because the goal is to rank existing content items for refresh review using observed performance and freshness signals. It can capture simple interactions between signals such as content staleness, CTR, impressions, and search position while remaining easy to inspect.

I am choosing a shallow tree rather than a complex ensemble because this is a baseline modeling exercise and interpretability is important. The model should provide useful decision support rather than add complexity without evidence of improvement.

The model will be compared with the Week-4 baseline on the same content items and using the same ranking metric.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I will use a grouped train/test split by `client_id`.

The same client will not appear in both the training and test sets. This is a more honest test of whether the model can rank content for clients it did not see during training.

I will use 80% of the clients for training and 20% for testing. The split is performed before model training to avoid client-level leakage.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("Grouped split check: PASS")

Dataset shape: (30000, 44)
Unique clients: 32

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0
Grouped split check: PASS


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I will train a shallow Decision Tree to rank content items using observed content and performance signals.

The target is whether the content has an observed `down` trend. This is used as a decision-support proxy for refresh prioritization, not as a causal label for whether refreshing will improve performance.

The model will use the same client-grouped test split defined in Section 2.

I will compare the model ranking with the Week-4 baseline using Precision@20 and Precision@50 on the same test rows. Higher precision means more of the highest-ranked items are observed as `down`.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Inspect candidate target signals before training.

print("Available columns:")
print(df.columns.tolist())

print("\nTrend direction distribution:")
display(
    df["trend_direction"]
    .value_counts(dropna=False)
    .rename_axis("trend_direction")
    .reset_index(name="n")
)

print("\nTrend percentage summary:")
print(df["trend_pct"].describe())

print("\nKey performance fields:")
display(
    df[
        [
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].describe()
)


Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Trend direction distribution:


,trend_direction,n
0,down,16262
1,stable,5962
2,up,4388
3,new,2236
4,flat,1152



Trend percentage summary:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Key performance fields:


,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update
count,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000
mean,5200.366300,16.097333,0.510733,16.34238,46.098300
std,16838.019547,75.076958,3.279162,15.21679,42.078709
min,1.000000,0.000000,0.000000,0.00000,1.000000
25%,81.000000,0.000000,0.000000,6.20000,20.000000
50%,731.000000,1.000000,0.070000,10.80000,20.000000
75%,3615.250000,7.000000,0.290000,22.30000,104.000000
max,517715.000000,4178.000000,100.000000,245.00000,373.000000


In [13]:
# Section 3: Train Decision Tree and compare with the Week-4 baseline.

from sklearn.tree import DecisionTreeClassifier
import pandas as pd

# Target:
# 1 = observed downward trend
# 0 = all other observed trend directions
train_df["target_down"] = (train_df["trend_direction"] == "down").astype(int)
test_df["target_down"] = (test_df["trend_direction"] == "down").astype(int)

# ---------------------------------------------------------
# 1. Recreate the Week-4 baseline ranking
# ---------------------------------------------------------

baseline_df = test_df.copy()

baseline_df["staleness_points"] = 0
baseline_df.loc[
    baseline_df["days_since_last_update"].between(91, 180),
    "staleness_points"
] = 1
baseline_df.loc[
    baseline_df["days_since_last_update"] > 180,
    "staleness_points"
] = 2

baseline_df["ctr_points"] = (
    baseline_df["ctr"] < 0.5
).astype(int)

baseline_df["baseline_score"] = (
    baseline_df["staleness_points"] +
    baseline_df["ctr_points"]
)

baseline_ranked = baseline_df.sort_values(
    ["baseline_score", "days_since_last_update", "ctr"],
    ascending=[False, False, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# 2. Train Decision Tree
# ---------------------------------------------------------

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X_train = train_df[features].fillna(0)
y_train = train_df["target_down"]

X_test = test_df[features].fillna(0)
y_test = test_df["target_down"]

model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=50,
    random_state=42
)

model.fit(X_train, y_train)

# Probability of observed downward trend.
test_df["model_score"] = model.predict_proba(X_test)[:, 1]

model_ranked = test_df.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

# ---------------------------------------------------------
# 3. Precision@K
# ---------------------------------------------------------

def precision_at_k(data, k):
    return data.head(k)["target_down"].mean()

baseline_p20 = precision_at_k(baseline_ranked, 20)
baseline_p50 = precision_at_k(baseline_ranked, 50)

model_p20 = precision_at_k(model_ranked, 20)
model_p50 = precision_at_k(model_ranked, 50)

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Decision Tree"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

# ---------------------------------------------------------
# 4. Results
# ---------------------------------------------------------

print("Target distribution in test set:")
print(
    test_df["target_down"]
    .value_counts()
    .rename({
        0: "not_down",
        1: "down"
    })
)

print("\nModel vs baseline:")
display(comparison)

print("\nDecision Tree top 20:")
display(
    model_ranked[
        [
            "content_id",
            "client_id",
            "model_score",
            "target_down",
            "days_since_last_update",
            "ctr",
            "impressions_90d"
        ]
    ].head(20)
)

Target distribution in test set:
target_down
down        3149
not_down    3014
Name: count, dtype: int64

Model vs baseline:


,method,Precision@20,Precision@50
0,Week-4 baseline,0.70,0.64
1,Decision Tree,0.45,0.66



Decision Tree top 20:


,content_id,client_id,model_score,target_down,days_since_last_update,ctr,impressions_90d
0,content_04a7a1b7b5ec,client_4e07408562,0.725377,0,13,0.00,487
1,content_304f48230142,client_f369cb89fc,0.725377,1,20,0.76,3803
2,content_e3de4ddde194,client_434c9b5ae5,0.725377,0,89,0.00,31
3,content_901b40631379,client_8527a891e2,0.725377,1,98,0.00,1
4,content_d389b3cd65c2,client_f369cb89fc,0.725377,0,20,2.58,349
5,content_b7ab61937a1c,client_f369cb89fc,0.725377,1,20,0.00,23
6,content_452a4e18212c,client_f369cb89fc,0.725377,1,106,0.04,2243
7,content_2ccae1f5c902,client_f369cb89fc,0.725377,0,20,0.00,73
8,content_81a89c686b96,client_f369cb89fc,0.725377,0,20,0.33,606
9,content_1a8b085d3f40,client_f369cb89fc,0.725377,0,20,0.00,151


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The Decision Tree was correct for 9 of its top 20 ranked items and wrong for 11. This matches the lower Precision@20 of 0.45 compared with 0.70 for the Week-4 baseline.

The tree relied most on `impressions_prev_30d`, followed by `content_age_days` and `impressions_last_30d`. This suggests that recent and previous-month impression levels were important for distinguishing observed downward-trending content in this model.

The model performed slightly better than the baseline at Precision@50 (0.66 vs 0.64), but it performed substantially worse at Precision@20. Therefore, the added model complexity did not provide a clear improvement over the simple baseline for the highest-priority review queue.

The errors show that the observed signals do not cleanly identify all downward-trending content. The model should therefore be treated as directional decision support rather than as a definitive refresh recommendation.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Inspect model features and Top-20 errors.

from sklearn.inspection import permutation_importance

# Built-in tree feature importance.
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Decision Tree feature importance:")
display(feature_importance.head(10))

# Inspect Top-20 errors.
top20_model = model_ranked.head(20).copy()
top20_model["correct"] = top20_model["target_down"] == 1

print("\nDecision Tree Top-20 errors:")
display(
    top20_model[
        [
            "content_id",
            "client_id",
            "model_score",
            "target_down",
            "days_since_last_update",
            "ctr",
            "impressions_90d"
        ]
    ]
)

print(
    f"\nTop-20 correct: {top20_model['correct'].sum()}/20"
)
print(
    f"Top-20 errors: {(~top20_model['correct']).sum()}/20"
)

Decision Tree feature importance:


,feature,importance
13,impressions_prev_30d,0.647172
16,content_age_days,0.169186
10,impressions_last_30d,0.134173
19,avg_position,0.049469
0,search_volume,0.000000
1,competition,0.000000
2,cpc,0.000000
6,clicks_90d,0.000000
5,impressions_90d,0.000000
4,char_count,0.000000



Decision Tree Top-20 errors:


,content_id,client_id,model_score,target_down,days_since_last_update,ctr,impressions_90d
0,content_04a7a1b7b5ec,client_4e07408562,0.725377,0,13,0.00,487
1,content_304f48230142,client_f369cb89fc,0.725377,1,20,0.76,3803
2,content_e3de4ddde194,client_434c9b5ae5,0.725377,0,89,0.00,31
3,content_901b40631379,client_8527a891e2,0.725377,1,98,0.00,1
4,content_d389b3cd65c2,client_f369cb89fc,0.725377,0,20,2.58,349
5,content_b7ab61937a1c,client_f369cb89fc,0.725377,1,20,0.00,23
6,content_452a4e18212c,client_f369cb89fc,0.725377,1,106,0.04,2243
7,content_2ccae1f5c902,client_f369cb89fc,0.725377,0,20,0.00,73
8,content_81a89c686b96,client_f369cb89fc,0.725377,0,20,0.33,606
9,content_1a8b085d3f40,client_f369cb89fc,0.725377,0,20,0.00,151



Top-20 correct: 9/20
Top-20 errors: 11/20


In [15]:
# Section 4: Compact error summary.

top20_model = model_ranked.head(20).copy()
top20_errors = top20_model[top20_model["target_down"] == 0].copy()

print("Decision Tree Top-20 error summary")
print(f"Top-20 items: {len(top20_model)}")
print(f"Correct: {top20_model['target_down'].sum()}")
print(f"Errors: {len(top20_errors)}")

print("\nModel vs baseline:")
display(comparison)

print("\nTop features used by the Decision Tree:")
display(feature_importance.head(5))

Decision Tree Top-20 error summary
Top-20 items: 20
Correct: 9
Errors: 11

Model vs baseline:


,method,Precision@20,Precision@50
0,Week-4 baseline,0.70,0.64
1,Decision Tree,0.45,0.66



Top features used by the Decision Tree:


,feature,importance
13,impressions_prev_30d,0.647172
16,content_age_days,0.169186
10,impressions_last_30d,0.134173
19,avg_position,0.049469
0,search_volume,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [16]:
# Final ML-08 self-check

forbidden_features = {
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
}

print("Feature leakage check:")
print("Forbidden features used:", forbidden_features.intersection(features))
assert not forbidden_features.intersection(features)

print("\nClient split check:")
print("Client overlap:", len(set(train_df["client_id"]) & set(test_df["client_id"])))
assert len(set(train_df["client_id"]) & set(test_df["client_id"])) == 0

print("\nRequired objects:")
print("Model:", type(model).__name__)
print("Comparison table:", comparison.shape)
print("Features used:", len(features))

assert isinstance(model, DecisionTreeClassifier)
assert comparison.shape[0] == 2
assert len(features) > 0

print("\nML-08 self-check: PASS")

Feature leakage check:
Forbidden features used: set()

Client split check:
Client overlap: 0

Required objects:
Model: DecisionTreeClassifier
Comparison table: (2, 3)
Features used: 23

ML-08 self-check: PASS
